# Extract Files from S3 bucket

In [8]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

AWS_PROFILE = os.getenv("AWS_PROFILE", "pavel-bot")
AWS_REGION = os.getenv("AWS_REGION", "eu-north-1")
S3_BUCKET = os.environ["S3_BUCKET_NAME"]

# Create a reusable S3 client
session = boto3.Session(profile_name=AWS_PROFILE)
s3 = session.client("s3", region_name=AWS_REGION)

In [10]:
def extract_markdown_files(s3_client, bucket_name: str) -> list[dict]:
    """
    Extract all Markdown files from S3 directly into memory.

    Returns a list of dictionaries containing:
        source_file: Relative path within the S3 bucket
        filename: Original filename
        content: Raw Markdown text
    """

    documents = []

    # Pagination ensures we can process more than 1,000 files
    paginator = s3_client.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket_name):
        for obj in page.get("Contents", []):

            key = obj["Key"]

            # Ignore objects that aren't Markdown files
            if not key.lower().endswith(".md"):
                continue

            # Read the object directly into memory
            response = s3_client.get_object(
                Bucket=bucket_name,
                Key=key
            )

            content = response["Body"].read().decode("utf-8-sig")

            documents.append({
                "source_file": key,
                "filename": key.rsplit("/", 1)[-1],
                "content": content
            })

    return documents

In [11]:
documents = extract_markdown_files(s3, S3_BUCKET)

print(f"Extracted {len(documents)} Markdown files.")

# Display file paths
for doc in documents:
    print(doc["source_file"])

Extracted 25 Markdown files.
knowledge/faq/recruiter-questions.md
knowledge/faq/technical-decisions.md
knowledge/profile/about.md
knowledge/profile/career-interests-and-working-style.md
knowledge/profile/education.md
knowledge/profile/links-and-contact.md
knowledge/profile/professional-experience.md
knowledge/profile/training-and-certifications.md
knowledge/projects/candle-quest.md
knowledge/projects/digisoft-rag-chatbot.md
knowledge/projects/earthquake-etl.md
knowledge/projects/facial-landmark-prediction.md
knowledge/projects/har-and-pain-detection.md
knowledge/projects/hi5.md
knowledge/projects/kumo-mochi.md
knowledge/projects/mlops-project.md
knowledge/projects/mood-recognition.md
knowledge/projects/portfolio-and-pavel-bot.md
knowledge/projects/propaganda-detection.md
knowledge/reference/repositories-and-sites.md
knowledge/skills/ai-and-ml.md
knowledge/skills/cloud-and-devops.md
knowledge/skills/data-engineering.md
knowledge/skills/embedded-and-research.md
knowledge/skills/programmi

In [12]:
# Examine the first extracted document
doc = documents[0]

print("Source:", doc["source_file"])
print("Filename:", doc["filename"])
print("\nContent:\n")
print(doc["content"][:1000])

Source: knowledge/faq/recruiter-questions.md
Filename: recruiter-questions.md

Content:

---
id: faq-recruiter-questions
title: Common Questions About Pavel
category: faq
tags: [recruiters, interview, career, strengths, projects, introduction]
---
# Common Recruiter Questions

## What is Pavel interested in professionally?

Data engineering, cloud and backend/software development are key current interests, alongside AI/ML and MLOps. He enjoys combining data processing, APIs, application functionality and deployment into one practical product.

## Has Pavel worked as a professional software engineer?

His technical background includes university research, self-directed projects, a deployed client website and a collaborative bootcamp application. He should not be described as having held a formal software engineer or data engineer position without separate evidence.

## What is a concrete example of Pavel delivering for a real client?

He built the Kumo Mochi website for a handmade desse

# Parse YAML frontmatter

In [14]:
import frontmatter

def parse_markdown_document(document: dict) -> dict:
    """
    Parse YAML frontmatter from a Markdown document.

    Args:
        document: Dictionary containing:
            - source_file
            - filename
            - content

    Returns:
        Dictionary containing parsed metadata and Markdown body.
    """

    # Extract YAML frontmatter and Markdown body
    parsed = frontmatter.loads(document["content"])

    # Extract metadata as a dictionary
    metadata = parsed.metadata

    # Validate essential metadata fields
    required_fields = ["id", "title", "category"]

    for field in required_fields:
        if not metadata.get(field):
            raise ValueError(
                f"Missing '{field}' in {document['source_file']}"
            )

    # Return a consistent structure for the next pipeline step
    return {
        "document_id": metadata["id"],
        "title": metadata["title"],
        "category": metadata["category"],
        "tags": metadata.get("tags", []),
        "status": metadata.get("status"),
        "links": metadata.get("links", {}),
        "source_file": document["source_file"],
        "content": parsed.content.strip()
    }

In [15]:
# Select the first document
sample_document = documents[0]

# Parse its metadata
parsed_document = parse_markdown_document(sample_document)

# Display metadata
for key, value in parsed_document.items():

    if key != "content":
        print(f"{key}: {value}")

print("\nMarkdown content preview:")
print(parsed_document["content"][:500])

document_id: faq-recruiter-questions
title: Common Questions About Pavel
category: faq
tags: ['recruiters', 'interview', 'career', 'strengths', 'projects', 'introduction']
status: None
links: {}
source_file: knowledge/faq/recruiter-questions.md

Markdown content preview:
# Common Recruiter Questions

## What is Pavel interested in professionally?

Data engineering, cloud and backend/software development are key current interests, alongside AI/ML and MLOps. He enjoys combining data processing, APIs, application functionality and deployment into one practical product.

## Has Pavel worked as a professional software engineer?

His technical background includes university research, self-directed projects, a deployed client website and a collaborative bootcamp applic


In [16]:
parsed_documents = []
failed_documents = []

for document in documents:
    try:
        parsed = parse_markdown_document(document)
        parsed_documents.append(parsed)

    except (ValueError, frontmatter.YAMLError) as error:
        failed_documents.append({
            "source_file": document["source_file"],
            "error": str(error)
        })

print(f"Successfully parsed: {len(parsed_documents)}")
print(f"Failed: {len(failed_documents)}")

for failure in failed_documents:
    print(failure)

Successfully parsed: 25
Failed: 0


In [ ]:
import pandas as pd

# Create a DataFrame for metadata without the full md bodies
metadata_df = pd.DataFrame([
    {
        key: value
        for key, value in doc.items()
        if key != "content"
    }
    for doc in parsed_documents
])

display(metadata_df)

# Chunk Documents

In [19]:
import re
import tiktoken

# Tokenizer suitable for OpenAI's text-embedding-3-small
encoding = tiktoken.get_encoding("cl100k_base")

TARGET_TOKENS = 350
MAX_TOKENS = 500
OVERLAP_TOKENS = 40

In [20]:
def count_tokens(text: str) -> int:
    """
    Count the number of tokens in a string.
    """
    return len(encoding.encode(text))

In [22]:
def split_markdown_sections(markdown: str) -> list[dict]:
    """
    Split Markdown into sections using headings (# through ######).

    Returns dictionaries containing:
        section: Full heading path
        content: Text belonging to that section
    """

    sections = []
    heading_stack = []
    current_lines = []

    def save_section():
        text = "\n".join(current_lines).strip()

        if text:
            sections.append({
                "section": " > ".join(
                    heading for _, heading in heading_stack
                ),
                "content": text
            })

    for line in markdown.splitlines():

        # Match Markdown headings, ignoring ordinary text
        match = re.match(r"^(#{1,6})\s+(.+?)\s*#*\s*$", line)

        if match:
            # Save the previous section before changing headings
            save_section()
            current_lines = []

            level = len(match.group(1))
            heading = match.group(2).strip()

            # Remove headings at the same level or deeper
            while heading_stack and heading_stack[-1][0] >= level:
                heading_stack.pop()

            heading_stack.append((level, heading))

        else:
            current_lines.append(line)

    # Save final section
    save_section()

    return sections

In [23]:
sample_doc = parsed_documents[0]

sections = split_markdown_sections(sample_doc["content"])

for section in sections:
    print("SECTION:", section["section"])
    print("TOKENS:", count_tokens(section["content"]))
    print("PREVIEW:", section["content"][:120])
    print("-" * 50)

SECTION: Common Recruiter Questions > What is Pavel interested in professionally?
TOKENS: 39
PREVIEW: Data engineering, cloud and backend/software development are key current interests, alongside AI/ML and MLOps. He enjoys
--------------------------------------------------
SECTION: Common Recruiter Questions > Has Pavel worked as a professional software engineer?
TOKENS: 42
PREVIEW: His technical background includes university research, self-directed projects, a deployed client website and a collabora
--------------------------------------------------
SECTION: Common Recruiter Questions > What is a concrete example of Pavel delivering for a real client?
TOKENS: 49
PREVIEW: He built the Kumo Mochi website for a handmade dessert business, including a React UI, Supabase-backed listings, admin f
--------------------------------------------------
SECTION: Common Recruiter Questions > Has Pavel led a team?
TOKENS: 49
PREVIEW: Yes, he acted as project leader/main developer in a four-person Hi

In [25]:
def split_long_text(
    text: str,
    max_tokens: int = MAX_TOKENS,
    overlap_tokens: int = OVERLAP_TOKENS
) -> list[str]:
    """
    Split text into token-limited chunks.

    Tries to preserve paragraphs.
    Falls back to token-level splitting for long paragraphs.
    """

    if max_tokens <= 0:
        raise ValueError("max_tokens must be positive")

    if not 0 <= overlap_tokens < max_tokens:
        raise ValueError("overlap_tokens must be smaller than max_tokens")

    if count_tokens(text) <= max_tokens:
        return [text.strip()] if text.strip() else []

    # Split at blank lines (paragraph boundaries)
    paragraphs = re.split(r"\n\s*\n", text.strip())

    chunks = []
    current = ""

    for paragraph in paragraphs:

        candidate = (
            f"{current}\n\n{paragraph}".strip()
            if current else paragraph
        )

        if count_tokens(candidate) <= max_tokens:
            current = candidate
            continue

        # Save previous chunk
        if current:
            chunks.append(current)
            current = ""

        # Split oversized individual paragraphs
        if count_tokens(paragraph) > max_tokens:

            tokens = encoding.encode(paragraph)
            step = max_tokens - overlap_tokens

            for start in range(0, len(tokens), step):
                token_slice = tokens[start:start + max_tokens]

                chunks.append(
                    encoding.decode(token_slice)
                )

        else:
            current = paragraph

    if current:
        chunks.append(current)

    return chunks

In [28]:
def chunk_document(document: dict) -> list[dict]:
    """
    Convert one parsed Markdown document into chunks.

    Preserves document metadata and section information.
    """

    sections = split_markdown_sections(document["content"])

    chunks = []
    chunk_index = 0

    for section in sections:

        # Split sections only when they exceed the maximum
        text_parts = split_long_text(
            section["content"],
            max_tokens=MAX_TOKENS,
            overlap_tokens=OVERLAP_TOKENS
        )

        for part in text_parts:

            # Heading context is included in the embedding text,
            # but the original chunk content stays separate.
            embedding_text = (
                f"{document['title']}\n"
                f"{section['section']}\n\n"
                f"{part}"
            ).strip()

            chunks.append({
                "document_id": document["document_id"],
                "source_file": document["source_file"],
                "chunk_index": chunk_index,

                "title": document["title"],
                "section": section["section"],
                "content": part,

                "category": document["category"],
                "tags": document["tags"],
                "status": document["status"],
                "links": document["links"],

                # Used by the embedding and hashing steps
                "embedding_text": embedding_text,
                "token_count": count_tokens(embedding_text)
            })

            chunk_index += 1

    return chunks

In [29]:
all_chunks = []

for document in parsed_documents:

    document_chunks = chunk_document(document)
    all_chunks.extend(document_chunks)

    print(
        f"{document['title']}: "
        f"{len(document_chunks)} chunks"
    )

print("\nTotal chunks:", len(all_chunks))

Common Questions About Pavel: 12 chunks
Project Architecture and Technical Decision Explanations: 7 chunks
About Pavel Jermolajev: 7 chunks
Career Interests, Motivation and Working Style: 5 chunks
University Education and Academic Foundations: 3 chunks
Public Professional Links and Contact: 3 chunks
Professional Experience and Career Transition: 4 chunks
Additional Training and Certificates: 3 chunks
Candle Quest — Educational Pygame Project: 1 chunks
DigiSoft — RAG Chatbot and Cloud Deployment: 6 chunks
USGS Earthquake Data ETL on Azure Databricks: 5 chunks
Facial Landmark Prediction: 3 chunks
Human Activity Recognition and Sensor Misalignment Research: 7 chunks
Hi5 — Professional Networking and Matching Web App: 6 chunks
Kumo Mochi — Client Website: 5 chunks
MLOps and Model Deployment Experiments: 4 chunks
Mood Recognition in Images: 3 chunks
Personal Portfolio and Pavel-Bot RAG Assistant: 5 chunks
Propaganda Detection and Classification with NLP: 4 chunks
Project Repositories and Li

In [30]:
import pandas as pd

chunk_df = pd.DataFrame([
    {
        "document_id": chunk["document_id"],
        "chunk_index": chunk["chunk_index"],
        "section": chunk["section"],
        "token_count": chunk["token_count"],
        "preview": chunk["content"][:100]
    }
    for chunk in all_chunks
])

display(chunk_df)

,document_id,chunk_index,section,token_count,preview
0,faq-recruiter-questions,0,Common Recruiter Questions > What is Pavel int...,56,"Data engineering, cloud and backend/software d..."
1,faq-recruiter-questions,1,Common Recruiter Questions > Has Pavel worked ...,61,His technical background includes university r...
2,faq-recruiter-questions,2,Common Recruiter Questions > What is a concret...,72,He built the Kumo Mochi website for a handmade...
3,faq-recruiter-questions,3,Common Recruiter Questions > Has Pavel led a t...,65,"Yes, he acted as project leader/main developer..."
4,faq-recruiter-questions,4,Common Recruiter Questions > What was Pavel do...,52,He spent nearly a decade working as a professi...
...,...,...,...,...,...
112,skills-embedded-research,3,Embedded Systems and Research > Project evidence,37,HAR and Pain Detection: https://github.com/Mis...
113,skills-programming-software,0,Programming and Software Development > Main la...,114,- **Python:** Pavel's primary language for mac...
114,skills-programming-software,1,Programming and Software Development > Web dev...,70,"Pavel has built applications with React, Vite,..."
115,skills-programming-software,2,Programming and Software Development > Backend...,65,"His experience includes Python applications, A..."


# Generate Content Hashes

In [32]:
import hashlib

def generate_content_hash(text: str) -> str:
    """
    Generate a SHA-256 hash from a text string.

    Args:
        text: The exact text that will be sent to the embedding model.

    Returns:
        A 64-character hexadecimal SHA-256 hash.
    """

    # Convert the string to bytes using UTF-8 encoding
    text_bytes = text.encode("utf-8")

    # Calculate the SHA-256 digest
    digest = hashlib.sha256(text_bytes)

    # Return its hexadecimal representation
    return digest.hexdigest()

In [35]:
for chunk in all_chunks:

    # Hash the exact text used for embedding, including title and heading
    chunk["content_hash"] = generate_content_hash(
        chunk["embedding_text"]
    )

In [37]:
assert all(
    len(chunk["content_hash"]) == 64
    for chunk in all_chunks
)

print("All chunks have valid-length SHA-256 hashes.")

All chunks have valid-length SHA-256 hashes.


In [38]:
from collections import Counter

hash_counts = Counter(
    chunk["content_hash"]
    for chunk in all_chunks
)

duplicates = {
    content_hash: count
    for content_hash, count in hash_counts.items()
    if count > 1
}

print("Duplicate content hashes:", len(duplicates))

Duplicate content hashes: 0


# Generate Embeddings

In [39]:
import os
import psycopg
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Initialise the OpenAI client.
client = OpenAI()

# Embedding model and expected output dimensions.
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSIONS = 1536

# Neon PostgreSQL connection string.
DATABASE_URL = os.environ["DATABASE_URL"]

In [41]:
def fetch_existing_hashes() -> dict:
    """
    Retrieve existing chunk hashes from Neon.

    Returns:
        Dictionary mapping:
        (document_id, chunk_index) -> content_hash
    """

    existing_hashes = {}

    # Open a PostgreSQL connection.
    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cursor:

            cursor.execute("""
                SELECT document_id, chunk_index, content_hash
                FROM knowledge_chunks
            """)

            # Read all existing records.
            for document_id, chunk_index, content_hash in cursor:

                existing_hashes[(document_id, chunk_index)] = content_hash

    return existing_hashes

existing_hashes = fetch_existing_hashes()

print(f"Existing database chunks: {len(existing_hashes)}")

Existing database chunks: 0


In [43]:
def identify_changed_chunks(
    chunks: list[dict],
    existing_hashes: dict
) -> list[dict]:
    """
    Return chunks that are new or have changed.

    Unchanged chunks are excluded from embedding generation.
    """

    changed_chunks = []

    for chunk in chunks:

        # Identify the chunk using its document ID and position.
        chunk_key = (
            chunk["document_id"],
            chunk["chunk_index"]
        )

        # Retrieve previously stored hash, if available.
        stored_hash = existing_hashes.get(chunk_key)

        # Include chunk only when its hash differs.
        if stored_hash != chunk["content_hash"]:
            changed_chunks.append(chunk)

    return changed_chunks


changed_chunks = identify_changed_chunks(
    all_chunks,
    existing_hashes
)

print("Total chunks:", len(all_chunks))
print("Need embedding:", len(changed_chunks))
print("Unchanged:", len(all_chunks) - len(changed_chunks))

Total chunks: 117
Need embedding: 117
Unchanged: 0


In [44]:
def generate_embedding(text: str) -> list[float]:
    """
    Generate a numerical embedding for one piece of text.

    Returns:
        A list of 1536 floating-point numbers.
    """

    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )

    return response.data[0].embedding

In [47]:
def generate_embeddings_batch(
    chunks: list[dict],
    batch_size: int = 50
) -> list[dict]:
    """
    Generate embeddings for a collection of chunks.

    Adds an 'embedding' field to each returned chunk.
    Processes chunks in batches to reduce API requests.
    """

    if batch_size < 1:
        raise ValueError("batch_size must be positive")

    embedded_chunks = []

    for start in range(0, len(chunks), batch_size):

        # Get the current batch.
        batch = chunks[start:start + batch_size]

        # Extract the exact text we previously hashed.
        texts = [
            chunk["embedding_text"]
            for chunk in batch
        ]

        # Request all embeddings in a single API call.
        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=texts
        )

        # Associate each embedding with its corresponding chunk.
        # The API returns an index for each input.
        vectors = {
            item.index: item.embedding
            for item in response.data
        }

        if len(vectors) != len(batch):
            raise ValueError("Embedding response count mismatch")

        for index, chunk in enumerate(batch):

            embedding = vectors[index]

            if len(embedding) != EMBEDDING_DIMENSIONS:
                raise ValueError(
                    f"Unexpected embedding size: {len(embedding)}"
                )

            # Create a new dictionary without changing the source.
            embedded_chunks.append({
                **chunk,
                "embedding": embedding
            })

        print(
            f"Embedded {len(embedded_chunks)}/{len(chunks)} chunks"
        )

    return embedded_chunks

In [48]:
embedded_chunks = generate_embeddings_batch(
    changed_chunks,
    batch_size=50
)

print(f"\nSuccessfully embedded: {len(embedded_chunks)}")

Embedded 50/117 chunks
Embedded 100/117 chunks
Embedded 117/117 chunks

Successfully embedded: 117


In [50]:
print(embedded_chunks[0].keys())

dict_keys(['document_id', 'source_file', 'chunk_index', 'title', 'section', 'content', 'category', 'tags', 'status', 'links', 'embedding_text', 'token_count', 'content_hash', 'embedding'])


# Insert or Update Neon Records


In [51]:
import json
import psycopg
from psycopg.types.json import Jsonb

In [52]:
def upsert_embedded_chunks(chunks: list[dict]) -> int:
    """
    Insert new chunks or update existing chunks in Neon.

    Uniqueness is determined by:
        (document_id, chunk_index)

    Returns:
        Number of chunks processed.
    """

    if not chunks:
        return 0

    query = """
        INSERT INTO knowledge_chunks (
            document_id,
            source_file,
            chunk_index,
            title,
            section,
            content,
            embedding,
            category,
            tags,
            status,
            links,
            content_hash
        )
        VALUES (
            %s, %s, %s, %s, %s, %s,
            %s::vector, %s, %s, %s, %s, %s
        )
        ON CONFLICT (document_id, chunk_index)
        DO UPDATE SET
            source_file = EXCLUDED.source_file,
            title = EXCLUDED.title,
            section = EXCLUDED.section,
            content = EXCLUDED.content,
            embedding = EXCLUDED.embedding,
            category = EXCLUDED.category,
            tags = EXCLUDED.tags,
            status = EXCLUDED.status,
            links = EXCLUDED.links,
            content_hash = EXCLUDED.content_hash
    """

    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cursor:

            for chunk in chunks:

                # Format embedding as pgvector-compatible text.
                vector = "[" + ",".join(
                    str(value) for value in chunk["embedding"]
                ) + "]"

                cursor.execute(
                    query,
                    (
                        chunk["document_id"],
                        chunk["source_file"],
                        chunk["chunk_index"],
                        chunk["title"],
                        chunk["section"],
                        chunk["content"],
                        vector,
                        chunk["category"],
                        chunk["tags"],
                        chunk["status"],
                        Jsonb(chunk["links"]),
                        chunk["content_hash"]
                    )
                )

        # Commit the transaction after all records succeed.
        conn.commit()

    return len(chunks)

In [54]:
inserted_count = upsert_embedded_chunks(embedded_chunks)

print(f"Successfully processed {inserted_count} chunks.")

Successfully processed 117 chunks.


### Update metadata for unchanged chunks

In [55]:
def update_chunk_metadata(chunks: list[dict]) -> int:
    """
    Update metadata for existing chunks without
    changing their embeddings or content hashes.

    Returns:
        Number of database records updated.
    """

    query = """
        UPDATE knowledge_chunks
        SET
            source_file = %s,
            title = %s,
            section = %s,
            content = %s,
            category = %s,
            tags = %s,
            status = %s,
            links = %s
        WHERE
            document_id = %s
            AND chunk_index = %s
            AND content_hash = %s
    """

    updated = 0

    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cursor:

            for chunk in chunks:

                cursor.execute(
                    query,
                    (
                        chunk["source_file"],
                        chunk["title"],
                        chunk["section"],
                        chunk["content"],
                        chunk["category"],
                        chunk["tags"],
                        chunk["status"],
                        Jsonb(chunk["links"]),
                        chunk["document_id"],
                        chunk["chunk_index"],
                        chunk["content_hash"]
                    )
                )

                updated += cursor.rowcount

        conn.commit()

    return updated

In [56]:
# Find chunks whose hashes already matched Neon.
unchanged_chunks = [
    chunk for chunk in all_chunks
    if existing_hashes.get((
        chunk["document_id"],
        chunk["chunk_index"]
    )) == chunk["content_hash"]
]

updated_count = update_chunk_metadata(unchanged_chunks)

print(f"Metadata updated for {updated_count} chunks.")

Metadata updated for 0 chunks.


## Delete non-existent data

In [58]:
def delete_obsolete_chunks(
    document_id: str,
    valid_indexes: list[int]
) -> int:
    """
    Remove chunks whose indexes are no longer present
    in the latest version of a document.

    Only affects the specified document.
    """

    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cursor:

            cursor.execute("""
                DELETE FROM knowledge_chunks
                WHERE document_id = %s
                  AND NOT (chunk_index = ANY(%s))
            """, (document_id, valid_indexes))

            deleted = cursor.rowcount

        conn.commit()

    return deleted

In [59]:
for document in parsed_documents:

    document_id = document["document_id"]

    valid_indexes = [
        chunk["chunk_index"]
        for chunk in all_chunks
        if chunk["document_id"] == document_id
    ]

    deleted = delete_obsolete_chunks(
        document_id,
        valid_indexes
    )

    if deleted:
        print(f"{document_id}: removed {deleted} obsolete chunks")